# 01 — Microsoft Fabric Spark Fundamentals

**Audience:** Data engineers new to Microsoft Fabric who already know PySpark basics.

**What this notebook covers**
- How a Fabric notebook session is created and attached to a Lakehouse
- Reading and writing data with the Lakehouse (Files vs Tables)
- Core DataFrame operations you'll use every day
- Multi-language cells (`%%pyspark`, `%%sql`, `%%csharp`, `%%r`)
- Basic Spark configuration inside a notebook

> Attach a Lakehouse to this notebook first (View > Lakehouse explorer > Add) before running the cells below.


## 1. The implicit Spark session

Fabric notebooks auto-create a `SparkSession` called `spark` — you never call `SparkSession.builder` yourself.

In [ ]:
# spark is already available in every Fabric notebook cell
print(spark.version)
print(spark.sparkContext.appName)

# See the current Spark configuration
for k, v in spark.sparkContext.getConf().getAll()[:10]:
    print(k, "=", v)


## 2. Reading data — Files vs Tables

A Lakehouse has two areas:
- **Files** — raw/unmanaged files (csv, json, parquet, delta folders you drop in yourself)
- **Tables** — managed Delta tables registered in the Lakehouse SQL endpoint

Both are addressable from Spark.

In [ ]:
# Reading a CSV sitting in the Lakehouse "Files" section
df_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("Files/raw/transactions/2026-01/*.csv")
)
df_raw.printSchema()
df_raw.show(5, truncate=False)


In [ ]:
# Reading a managed Delta table by name (no path needed)
df_customers = spark.read.table("customers")

# Equivalent using the ABFSS path (useful for cross-workspace / cross-lakehouse reads)
df_customers_path = spark.read.format("delta").load(
    "abfss://<workspace-id>@onelake.dfs.fabric.microsoft.com/<lakehouse-id>/Tables/customers"
)

df_customers.show(5)


## 3. Building DataFrames programmatically

In [ ]:
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType

schema = StructType([
    StructField("txn_id", StringType(), False),
    StructField("account_id", StringType(), False),
    StructField("amount", DoubleType(), True),
    StructField("txn_ts", TimestampType(), True),
])

sample_data = [
    Row(txn_id="T1001", account_id="A100", amount=2500.00, txn_ts="2026-01-05 10:15:00"),
    Row(txn_id="T1002", account_id="A101", amount=-450.75, txn_ts="2026-01-05 10:16:22"),
]

df_sample = spark.createDataFrame(sample_data, schema=schema)
df_sample.show()


## 4. Everyday DataFrame operations

In [ ]:
from pyspark.sql import functions as F

df_txn = df_raw

df_summary = (
    df_txn
    .withColumn("txn_month", F.date_format("txn_date", "yyyy-MM"))
    .filter(F.col("amount").isNotNull())
    .groupBy("txn_month", "branch_code")
    .agg(
        F.count("*").alias("txn_count"),
        F.sum("amount").alias("total_amount"),
        F.round(F.avg("amount"), 2).alias("avg_amount"),
    )
    .orderBy("txn_month", F.desc("total_amount"))
)

df_summary.show(20, truncate=False)


## 5. Writing results back to the Lakehouse

In [ ]:
(
    df_summary.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver_txn_monthly_summary")
)

# Or write raw files instead of a managed table
(
    df_summary.write
    .mode("overwrite")
    .parquet("Files/curated/txn_monthly_summary")
)


## 6. Multi-language cells

Fabric notebooks let you mix languages in the same notebook — each cell shares the same Lakehouse context (but not Python variables across languages).

In [ ]:
%%sql
-- Runs against the Lakehouse SQL context; shares the metastore with the PySpark cells above
SELECT branch_code, SUM(total_amount) AS branch_total
FROM silver_txn_monthly_summary
GROUP BY branch_code
ORDER BY branch_total DESC


In [ ]:
%%configure -f
{
    "defaultLakehouse": {
        "name": "SalesLakehouse"
    },
    "conf": {
        "spark.sql.shuffle.partitions": "50"
    }
}
# %%configure must be the FIRST code cell run in a session — it restarts the Spark session
# with the given configuration/attached lakehouse.


## 7. Quick sanity checklist before moving on
- [ ] Lakehouse attached to the notebook
- [ ] `spark.read.table(...)` and `spark.read.csv(...)` both work
- [ ] You can write a Delta table with `saveAsTable`
- [ ] `%%sql` cells can query tables you just wrote

Next notebook: **02 — Lakehouse & Delta Lake Deep Dive**.